In [16]:
import cv2
import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import local_binary_pattern

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [17]:
def extract_lbp(face_gray):
    face = cv2.resize(face_gray, (128, 128))
    blurred = cv2.GaussianBlur(face, (5, 5), 1.0)

    P = 8
    R = 1
    METHOD = "uniform"

    lbp = local_binary_pattern(blurred, P, R, METHOD)

    n_bins = P + 2
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=n_bins,
        range=(0, n_bins)
    )

    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return hist


In [18]:
def extract_laplacian(face_gray):
    face = cv2.resize(face_gray, (128, 128))
    blurred = cv2.GaussianBlur(face, (3, 3), 0)

    edges = cv2.Canny(face, 50, 150)
    lap = cv2.Laplacian(blurred, cv2.CV_64F)
    abs_lap = np.abs(lap)

    masked = abs_lap[edges > 0]

    if len(masked) == 0:
        return np.zeros(3)

    mean = masked.mean()
    std  = masked.std()
    var  = masked.var()

    return np.array([mean, std, var])


In [19]:
def process_data(face_gray):
    lbp_feat = extract_lbp(face_gray)
    lap_feat = extract_laplacian(face_gray)

    feature = np.concatenate([lbp_feat, lap_feat])

    return feature


In [20]:
def load_data(folder):
    X, y = [], []

    color_dir = os.path.join(folder, "color")

    for file in os.listdir(color_dir):
        if not file.endswith(".jpg"):
            continue

        label = 1 if "real" in file.lower() else 0

        img_path = os.path.join(color_dir, file)
        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.3, minNeighbors=5
        )

        if len(faces) == 0:
            continue

        x, y0, w, h = faces[0]
        face = gray[y0:y0+h, x:x+w]

        feature = process_data(face)

        X.append(feature)
        y.append(label)

    return np.array(X), np.array(y)

In [21]:
def train_model(X_train, y_train, model_path="rf_lbp_lap_ver2.pkl"):
    clf = RandomForestClassifier(
        n_estimators=600,
        max_depth=40,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train)

    joblib.dump(clf, model_path)
    print(f"Model saved to: {model_path}")

    return clf

In [22]:
train_folder = "data/train_img/train_img"
test_folder  = "data/test_img/test_img"

print("Loading train data...")
X_train, y_train = load_data(train_folder)

print("Loading test data...")
X_test, y_test = load_data(test_folder)

print("Training model...")
model = train_model(X_train, y_train)

print("Evaluating...")
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Loading train data...
Loading test data...
Training model...
Model saved to: rf_lbp_lap_ver2.pkl
Evaluating...
Accuracy: 0.8025210084033614
              precision    recall  f1-score   support

           0       0.80      0.98      0.88      1601
           1       0.80      0.29      0.42       541

    accuracy                           0.80      2142
   macro avg       0.80      0.63      0.65      2142
weighted avg       0.80      0.80      0.77      2142



In [29]:
import cv2
import numpy as np
import joblib
from skimage.feature import local_binary_pattern


class FaceAntiSpoofing:
    def __init__(
        self,
        model_path,
        cam_id=0,
        fake_threshold=0.90,
        display_size=(480, 640),
        window_name="Face Anti-Spoofing",
        show_features=True,
    ):
        self.model = joblib.load(model_path)
        print("Model loaded:", model_path)

        self.fake_threshold = fake_threshold
        self.display_h, self.display_w = display_size
        self.window_name = window_name
        self.show_features = show_features

        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        )

        self.cap = cv2.VideoCapture(cam_id)
        if not self.cap.isOpened():
            raise RuntimeError("Cannot open camera")

        self.active = False  # toggled by spacebar instead of auto-start

    def _extract_lbp(self, face_gray, return_map=False):
        face = cv2.resize(face_gray, (128, 128))
        blurred = cv2.GaussianBlur(face, (5, 5), 1.0)

        P, R = 8, 1
        lbp = local_binary_pattern(blurred, P, R, method="uniform")

        n_bins = P + 2
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=n_bins,
            range=(0, n_bins)
        )

        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-6)

        if return_map:
            return hist, lbp
        return hist

    def _extract_laplacian(self, face_gray, return_map=False):
        face = cv2.resize(face_gray, (128, 128))
        blurred = cv2.GaussianBlur(face, (3, 3), 0)

        edges = cv2.Canny(face, 50, 150)
        lap = cv2.Laplacian(blurred, cv2.CV_64F)
        abs_lap = np.abs(lap)

        masked = abs_lap[edges > 0]
        if len(masked) == 0:
            stats = np.zeros(3)
        else:
            stats = np.array([
                masked.mean(),
                masked.std(),
                masked.var()
            ])

        if return_map:
            return stats, abs_lap, edges
        return stats

    def _extract_feature(self, face_gray):
        lbp_feat = self._extract_lbp(face_gray)
        lap_feat = self._extract_laplacian(face_gray)
        return np.concatenate([lbp_feat, lap_feat])

    def _predict(self, face_gray):
        feature = self._extract_feature(face_gray).reshape(1, -1)
        pred = self.model.predict(feature)[0]
        prob = self.model.predict_proba(feature)[0]
        return pred, np.max(prob)

    def _hist_to_image(self, hist, size=(640, 150)):
        h, w = size[1], size[0]
        canvas = np.zeros((h, w, 3), dtype=np.uint8)

        canvas[:] = (18, 18, 18)
        cv2.rectangle(canvas, (0, 0), (w - 1, h - 1), (42, 42, 42), 1)
        cv2.line(canvas, (40, h - 35), (w - 12, h - 35), (75, 75, 75), 1)

        if hist.max() > 0:
            bin_w = max(2, (w - 70) // len(hist))
            max_val = hist.max()
            for i, val in enumerate(hist):
                bar_h = int((val / max_val) * (h - 55))
                x0 = 40 + i * bin_w
                x1 = min(w - 18, x0 + bin_w - 4)
                cv2.rectangle(canvas, (x0, h - 36), (x1, h - 36 - bar_h), (0, 190, 70), -1)

        cv2.putText(canvas, "LBP histogram", (14, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (210, 210, 210), 2)
        return canvas

    def _compose_diag_panel(self, visuals):
        panel_w = self.display_w
        panel_h = self.display_h

        # Remove title, tighten padding, make thumbnails larger and square
        info_h = 52
        hist_h = 130
        pad = 4

        content_w = int(panel_w * 0.85)
        x_margin = (panel_w - content_w) // 2

        available_h = panel_h - info_h - hist_h - pad * 3
        available_h = max(140, available_h)
        cell_size = min(
            (content_w - pad * 3) // 2,
            (available_h - pad) // 2,
        )
        cell_size = max(120, cell_size)

        grid_w = 2 * cell_size + pad
        grid_h = 2 * cell_size + pad
        grid_x0 = x_margin + max(0, (content_w - grid_w) // 2)
        grid_y0 = pad

        canvas = np.zeros((panel_h, panel_w, 3), dtype=np.uint8)
        canvas[:] = (12, 12, 12)

        # Grid of four views (square thumbnails)
        slots = [
            (visuals.get("gray"), "Gray face"),
            (visuals.get("lbp_map"), "LBP map"),
            (visuals.get("edges"), "Canny edges"),
            (visuals.get("laplacian"), "Laplacian abs"),
        ]
        for idx, (img, title) in enumerate(slots):
            if img is None:
                continue
            r, c = divmod(idx, 2)
            y0 = grid_y0 + r * (cell_size + pad)
            x0 = grid_x0 + c * (cell_size + pad)
            thumb = cv2.resize(img, (cell_size, cell_size))
            if thumb.ndim == 2:
                thumb = cv2.cvtColor(thumb, cv2.COLOR_GRAY2BGR)
            cv2.rectangle(canvas, (x0, y0), (x0 + cell_size, y0 + cell_size), (35, 35, 35), 1)
            canvas[y0:y0 + cell_size, x0:x0 + cell_size] = thumb
            cv2.rectangle(canvas, (x0, y0), (x0 + 180, y0 + 26), (0, 0, 0), -1)
            cv2.putText(canvas, title, (x0 + 10, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        # Info strip (feature text) under grid, full width, dark background and border
        info_y0 = grid_y0 + grid_h + pad
        x_info = 0
        info_w = panel_w
        cv2.rectangle(canvas, (x_info, info_y0), (x_info + info_w - 1, info_y0 + info_h), (12, 12, 12), -1)
        cv2.rectangle(canvas, (x_info, info_y0), (x_info + info_w - 1, info_y0 + info_h), (12, 12, 12), 1)

        stats = visuals.get("lap_stats")
        feat_text = visuals.get("feature_text")
        if stats is not None:
            lap_line = f"Lap mean {stats[0]:.2f} | std {stats[1]:.2f} | var {stats[2]:.2f}"
            cv2.putText(canvas, lap_line, (x_info + 10, info_y0 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.66, (180, 220, 255), 2)

        if self.show_features and feat_text is not None:
            lbp_line = feat_text.get('lbp_line', '')
            cv2.putText(canvas, lbp_line, (x_info + 10, info_y0 + 42), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (200, 240, 200), 1)

        # Histogram at bottom, full width, dark background
        hist_img = visuals.get("lbp_hist")
        hist_y0 = info_y0 + info_h + pad
        if hist_img is not None:
            hist_resized = cv2.resize(hist_img, (panel_w, hist_h))
            x_hist = 0
            canvas[hist_y0: hist_y0 + hist_h, x_hist:x_hist + panel_w] = hist_resized

        return canvas

    def _prepare_visuals(self, face_gray):
        hist, lbp_map = self._extract_lbp(face_gray, return_map=True)
        lap_stats, lap_map, edges = self._extract_laplacian(face_gray, return_map=True)

        lbp_norm = (lbp_map - lbp_map.min()) / (lbp_map.max() - lbp_map.min() + 1e-6)
        lbp_vis = (lbp_norm * 255).astype(np.uint8)

        lap_vis = cv2.normalize(lap_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        edges_vis = edges

        hist_img = self._hist_to_image(hist, size=(self.display_w, 150))

        face_resized = cv2.resize(face_gray, (self.display_w // 2, self.display_h // 2))

        lbp_first = hist[:10]
        lbp_vals = [f"{v:.3f}" for v in lbp_first]
        lbp_line = "LBP[:10]: " + ",".join(lbp_vals)

        return {
            "gray": face_resized,
            "lbp_map": lbp_vis,
            "edges": edges_vis,
            "laplacian": lap_vis,
            "lbp_hist": hist_img,
            "lap_stats": lap_stats,
            "feature_text": {"lbp_line": lbp_line},
        }

    def run(self):
        print("Press SPACE to start/pause, 'f' toggle features, 'q' to quit")

        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            frame_disp = cv2.resize(frame, (self.display_w, self.display_h))

            diag_panel = np.zeros_like(frame_disp)
            cv2.putText(frame_disp, "SPACE: start/stop | f: toggle features | q: quit", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            if self.active:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = self.face_cascade.detectMultiScale(
                    gray,
                    scaleFactor=1.3,
                    minNeighbors=5
                )

                for (x, y, w, h) in faces:
                    face_gray = gray[y:y + h, x:x + w]

                    pred, conf = self._predict(face_gray)

                    if pred == 0 and conf >= self.fake_threshold:
                        label = f"FAKE ({conf:.2f})"
                        color = (0, 0, 255)
                    else:
                        label = f"REAL ({conf:.2f})"
                        color = (0, 255, 0)

                    cv2.rectangle(frame_disp, (x, y), (x + w, y + h), color, 2)
                    cv2.putText(
                        frame_disp,
                        label,
                        (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        color,
                        2
                    )

                    visuals = self._prepare_visuals(face_gray)
                    diag_panel = self._compose_diag_panel(visuals)
                    break  # use first face for diagnostics

            else:
                cv2.putText(frame_disp, "Press SPACE to start", (15, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

            combined = np.hstack([frame_disp, diag_panel])
            cv2.imshow(self.window_name, combined)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key == 32:  # space
                self.active = not self.active
            if key == ord('f'):
                self.show_features = not self.show_features

        self.cap.release()
        cv2.destroyAllWindows()


In [30]:
fas = FaceAntiSpoofing(
    model_path="rf_lbp_lap.pkl",
    fake_threshold=0.8,
    display_size=(480, 640),
    window_name="Face Anti-Spoofing UI"
)
fas.run()


Model loaded: rf_lbp_lap.pkl
Press SPACE to start/pause, 'f' toggle features, 'q' to quit
